# Tree of Thought

PES2UG23CS928

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

In [ ]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [ ]:
def generate_text(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generate_text("Solve step by step: What is 12 * 8?"))

28 degrees


In [ ]:
def generate_text(prompt, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def generate_thoughts(question, k=3):
    thoughts = []
    for i in range(k):
        prompt = f"""
Question: {question}
Think step by step and suggest one possible reasoning path.
Thought:
"""
        thought = generate_text(prompt)
        thoughts.append(thought)
    return thoughts

In [ ]:
def expand_thought(thought):
    prompt = f"""
Here is a reasoning step:
{thought}

Continue this reasoning logically:
"""
    return generate_text(prompt)

In [ ]:
def score_thought(thought):
    score = len(thought)
    logic_keywords = ["because", "therefore", "so", "hence", "thus"]

    for word in logic_keywords:
        if word in thought.lower():
            score += 20
    return score

In [ ]:
def tree_of_thought_solver(question, branches=3):
    print("Generating initial thoughts...\n")
    thoughts = generate_thoughts(question, branches)

    expanded = []
    for t in thoughts:
        e = expand_thought(t)
        expanded.append(e)

    scored = [(score_thought(t), t) for t in expanded]
    scored.sort(reverse=True, key=lambda x: x[0])

    best_thought = scored[0][1]

    final_prompt = f"""
Based on the reasoning below, give the final answer clearly.

Reasoning:
{best_thought}

Final Answer:
"""

    final_answer = generate_text(final_prompt)
    return final_answer

In [ ]:
question = "If a train travels 60 km in 1 hour and then 30 km in 30 minutes, what is its average speed?"

answer = tree_of_thought_solver(question)
print("Final Answer:\n", answer)

Generating initial thoughts...

Final Answer:
 One way you get up top will work 1000 sq mi since 4 kg to weigh you back then 500 on 35000rnd / 1500 so 100p(25 sec). 2/5th your 4150 km time this makes 200 per se * 16 * 2000 time it then 1 hour long because 2000/170s would have 400mph, 1600rwipm + 1800n/v* 
